# TradeSense — EDA & Target Definitions

Exploratory analysis of historical OHLCV data and construction of ML target labels under the TradeSense execution convention.

- **Universe:** AAPL, MSFT, JPM, XOM, SPY (configurable)
- **Date range:** 2021-01-01 to 2026-08-01 (configurable, not hard-coded)
- **Caching:** `data/processed/{SYMBOL}.csv` (gitignored)
- **Figures:** `figures/` (gitignored)

## Execution / target convention

Features use info **at or before close of session t**. Signal generated **after close t**. Position executed at **open of session t+1**. Targets therefore anchor at `open_{t+1}` and never assume execution at the signal close.

Primary target (V1): next-session direction of the open-to-close move, with neutral-zone epsilon = 0.001. See `docs/eda_analysis.md` for full math.

In [ ]:
from datetime import date
from pathlib import Path

UNIVERSE = ["AAPL", "MSFT", "JPM", "XOM", "SPY"]
START = date(2021, 1, 1)
END = date(2026, 8, 1)
CACHE_DIR = Path("data/processed")
FIGURES_DIR = Path("figures")

print(f"Universe: {UNIVERSE}\nRange: {START} -> {END}")

In [ ]:
import pandas as pd
from src.data import DataPipeline
from src.data.providers import YahooFinanceProvider
from src.analysis import candles_to_dataframe, save_candles_csv, load_candles_csv

pipeline = DataPipeline(YahooFinanceProvider())
frames = {}
for symbol in UNIVERSE:
    cache = CACHE_DIR / f"{symbol}.csv"
    if cache.exists():
        frames[symbol] = load_candles_csv(cache)
        print(f"{symbol}: loaded from cache ({len(frames[symbol])} rows)")
    else:
        result = pipeline.fetch_historical(symbol, START, END)
        df = candles_to_dataframe(result.candles)
        save_candles_csv(df, cache)
        frames[symbol] = df
        print(f"{symbol}: fetched {len(df)} rows, cached to {cache}")

long_df = pd.concat(
    [df.assign(symbol=sym) for sym, df in frames.items()], ignore_index=True
)
long_df.head()

In [ ]:
from src.analysis import assess_quality

for symbol, df in frames.items():
    report = assess_quality(df, symbol=symbol)
    print(report.summary())
    print()

In [ ]:
from src.analysis import (
    log_returns,
    rolling_volatility,
    drawdown_series,
    max_drawdown,
    describe_returns,
    autocorrelation,
)

aapl = frames["AAPL"]
close = aapl["close"]
ret = log_returns(close)

print(describe_returns(ret).round(4).to_string())
print(f"\nMax drawdown (AAPL): {max_drawdown(close):.2%}")
print("Autocorrelation of daily log returns (lags 1-5):")
print(autocorrelation(ret, lags=5).round(4).to_string())

In [ ]:
from src.analysis import (
    next_session_direction,
    forward_return,
    future_realized_volatility,
    chronological_split,
)

# Targets strictly follow the execution convention (features <= t, labels > t)
y_dir = next_session_direction(aapl)                    # V1 primary
n = 5
y_fwd = forward_return(aapl, n=n)                       # secondary
rvol = future_realized_volatility(aapl, n=n)            # secondary

print("Direction label value counts:")
print(y_dir.value_counts(dropna=False).sort_index().to_string())
print(f"\nForward-{n}d return (tail):"); print(y_fwd.tail().to_string())
print(f"\nFuture realized vol (tail):"); print(rvol.tail().round(5).to_string())

# Temporal splits (chronological, with purge gap >= horizon)
train, val, test = chronological_split(close.index, ratios=(0.7, 0.15, 0.15), gap=n)
print(f"\nChronological split: train {len(train)}, val {len(val)}, test {len(test)}")
print(f"Val range: {val.min()} .. {val.max()}")

In [ ]:
from src.analysis import plots

plots.plot_price(close, out_path=FIGURES_DIR / "aapl_price.png")
plots.plot_returns(ret, out_path=FIGURES_DIR / "aapl_returns.png")
plots.plot_return_distribution(ret, out_path=FIGURES_DIR / "aapl_return_dist.png")
plots.plot_rolling_volatility(
    rolling_volatility(ret, window=21), out_path=FIGURES_DIR / "aapl_rolling_vol.png"
)
plots.plot_volume(aapl["volume"], out_path=FIGURES_DIR / "aapl_volume.png")
plots.plot_drawdown(drawdown_series(close), out_path=FIGURES_DIR / "aapl_drawdown.png")
plots.plot_acf(autocorrelation(ret, lags=20), out_path=FIGURES_DIR / "aapl_acf.png")
print(f"Figures saved to {FIGURES_DIR}")

## Notes

- Adjusted prices (`auto_adjust=True`) are used for return continuity; they are research prices, **not necessarily executable market prices**. Backtesting must later distinguish the two.
- The ML layer (features, models, strategies, backtesting, risk) is intentionally **not** implemented in this phase.
- Do not commit cached data (`data/processed/`) or figures (`figures/`) — both are gitignored.